# Lab21 — GSM8K B2/B3 on Kaggle

Upload `Lab21_Kaggle_GSM8K.zip` as a **private Dataset**, add it as Input, enable **Internet** and select an available GPU (T4 x2 recommended; the code uses GPU 0 only).

Run cells in order. This is a separate Kaggle math run; existing Colab CSKH and math files on Drive are untouched. Baselines are measured before training in the Kaggle software/GPU environment. Input files are copied from read-only `/kaggle/input` into writable `/kaggle/working`.

The corpus has 250 train, 50 val, 50 eval, and 15 regression probes. Both masks use the same model revision, data, rank16, alpha32, LR1e-4 and 32 steps. Valid traces require a nonempty closed thinking block; the known opening tag in the generation prompt is accounted for.

Results/adapter exports are in `/kaggle/working`. Save a version **with output files** and download the ZIP before ending the session. Runtime storage alone is not a permanent backup. No full merged model is exported.

Public GSM8K may have been in Qwen pretraining. This is a learner-curated format/mask experiment, not evidence of novel private data or a full official GSM8K benchmark score.


In [ ]:
# 1. Load private input and prepare writable experiment folder
import os, sys, shutil, json, zipfile, subprocess, re
from pathlib import Path
# Use one GPU to keep effective batch=16, including on a T4 x2 runtime.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HOME'] = '/tmp/lab21_hf_cache'
WORK = Path('/kaggle/working/lab21_kaggle')
INPUT = Path('/kaggle/input')
if not (WORK / 'run_math.py').exists():
    archives = list(INPUT.rglob('Lab21_Kaggle_GSM8K.zip'))
    if archives:
        with zipfile.ZipFile(archives[0]) as z:
            assert all(name.startswith('lab21_kaggle/') for name in z.namelist())
            z.extractall('/kaggle/working')
    else:
        candidates = [p.parent for p in INPUT.rglob('run_math.py')
                      if (p.parent / 'src/labkit').is_dir()
                      and (p.parent / 'bonus_gsm8k/results/dataset_manifest.json').exists()]
        assert candidates, 'Add Input: upload Lab21_Kaggle_GSM8K.zip as a private dataset first.'
        shutil.copytree(candidates[0], WORK)
    for p in WORK.rglob('*'):
        p.chmod(0o755 if p.is_dir() else 0o644)
MATH_DIR = WORK / 'bonus_gsm8k'
os.chdir(WORK)
manifest = json.loads((MATH_DIR / 'results/dataset_manifest.json').read_text())
print('Prepared dataset:', manifest['sizes'])
print('Writable results:', MATH_DIR / 'results')


In [ ]:
# 2. Dependencies — keep Kaggle's working CUDA-enabled torch
from importlib.metadata import version
import torch
assert torch.cuda.is_available(), 'Settings > Accelerator: select GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
requirements = []
for line in (WORK / 'requirements.txt').read_text().splitlines():
    name = re.match(r'^\s*([A-Za-z0-9_.-]+)', line)
    # B2/B3 use 16-bit LoRA, not the optional 4-bit contrast.
    if name and name[1].lower() not in {'torch', 'bitsandbytes'}:
        requirements.append(line)
req = WORK / 'requirements-kaggle.txt'
req.write_text('\n'.join(requirements) + '\n')
constraints = WORK / 'torch-constraint.txt'
constraints.write_text('torch==' + version('torch') + '\n')
print('Installing training libraries; preserving existing torch...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req),
                '-c', str(constraints)], check=True)
print('Dependencies ready.')


In [ ]:
# 3. Freeze Kaggle math baselines BEFORE training

def run_visible(stage):
    command = [sys.executable, '-u', str(WORK / 'run_math.py'),
               '--home', str(MATH_DIR), '--stage', stage]
    with (WORK / (stage + '.log')).open('a') as log:
        p = subprocess.Popen(command, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        if p.wait() != 0:
            raise RuntimeError('Stage failed: ' + stage + '. Read the log above.')

run_visible('baseline')


In [ ]:
# 4. Train/evaluate the two matched mask runs, then summarize
run_visible('all')


In [ ]:
# 5. Read the summary and create a small downloadable result ZIP
summary = MATH_DIR / 'results/math_bonus_summary.json'
print(summary.read_text())
archive = Path('/kaggle/working/Lab21_Kaggle_BonusResults.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('data', 'results', 'adapters', 'scripts'):
        for p in (MATH_DIR / folder).rglob('*'):
            if p.is_file() and not any(part.startswith('checkpoint-') for part in p.parts):
                z.write(p, Path('bonus_gsm8k') / p.relative_to(MATH_DIR))
    z.write(WORK / 'run_math.py', 'run_math.py')
print('Result archive:', archive)
print('Save Version with output files, then download the result ZIP from Output.')
